<a href="https://colab.research.google.com/github/Jasp3r16/00_thesis_generative_timber/blob/main/24_25_optimizer_workflow_with_cost_and_ILP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 24_25_optimizer_workflow_with_cost_and_ILP  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-27
### Properties script
**Goal:** to generate a cost matrix for the geometry with use of the timber datasets, then using ILP to find the best matches   
**Inputs:**
*   CSV timber dataset
*   Digital geometry

**Outputs:**
*   Best match for each element in a structure

## Mounting Google drive

In [ ]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definieer je paden (Pas de naam 'Thesis_Project' aan naar jouw mapnaam)
BASE_PATH = '/content/drive/MyDrive/Thesis_TU/'
DATA_PATH = os.path.join(BASE_PATH, 'data_thesis/')
DATA_IMPORT_PATH = os.path.join(BASE_PATH, 'data_import/')
SCRIPT_PATH = os.path.join(BASE_PATH, 'notebooks_thesis/')
OUTPUT_PATH = os.path.join(BASE_PATH, 'research_exports/')

# 3. Voeg de Scripts map toe aan het systeem-pad
# Hierdoor kun je .py bestanden uit die map 'importen'
if SCRIPT_PATH not in sys.path:
    sys.path.append(SCRIPT_PATH)

print(f"✅ Drive mounted. Project directory: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Project directory: /content/drive/MyDrive/Thesis_TU/


# IMPORTING

## Dataset

In [ ]:
import pandas as pd

# Laad de geëxporteerde dataset in je nieuwe omgeving
# Voeg sep=';' toe aan je inlaad-functie!
df_input_stock = pd.read_csv(DATA_PATH + 'complete_timber.csv', sep=';')

# Print de kolommen om zeker te weten dat ze goed gesplitst zijn
print("Gevonden kolommen:", df_input_stock.columns.tolist())

# Controleer of alle data, inclusief de RS_0000 ID's en binaire states, goed is overgekomen
print("\nDataset succesvol ingeladen. Hier zijn de eerste elementen:\n")
print(df_input_stock.head())

Gevonden kolommen: ['Member_ID', 'State', 'Length_Actual', 'Depth', 'Width', 'E_modulus_eff', 'f_mk', 'Density', 'Embodied Carbon Coëfficiënt', 'Transport_Dist', 'Emmisiefactor', 'Bewerkingsfactor']

Dataset succesvol ingeladen. Hier zijn de eerste elementen:

  Member_ID  State  Length_Actual  Depth  Width  E_modulus_eff  f_mk  Density  \
0  NS_00000      0         2400.0  100.0   38.0        11000.0    24      420   
1  NS_00001      0         2400.0  100.0   50.0        11000.0    24      420   
2  NS_00002      0         2400.0  100.0   75.0        11000.0    24      420   
3  NS_00003      0         2400.0  100.0  100.0        11000.0    24      420   
4  NS_00004      0         2400.0  150.0   38.0        11000.0    24      420   

   Embodied Carbon Coëfficiënt  Transport_Dist  Emmisiefactor  \
0                        150.0             500         0.1473   
1                        150.0             500         0.1019   
2                        150.0             500         0.

## Search space

De search space wordt vanuit het geometrie script geimporteerd, dan wordt een willekeurige samenstelling gekozen als beginpunt van de optimalisatie.


In [ ]:
import json

json_path = DATA_IMPORT_PATH + 'search_space.json'
# Lees het "contract" in voor je optimizer
with open(json_path, 'r') as f:
    optimizer_search_space = json.load(f)

print(f"✅ Search Space ingeladen! De optimizer heeft {len(optimizer_search_space)} parameters om aan te draaien.")
# Nu kun je deze variabele direct aan je Optuna, PyTorch of Genetisch Algoritme voeren!

✅ Search Space ingeladen! De optimizer heeft 18 parameters om aan te draaien.


Dit script is bedoeld voor je Colab-omgeving. Het leest de search_space.json in, stelt de parameters dynamisch in op basis van jouw regels, en gebruikt een "dummy" voorspelling (waar later je echte Neurale Netwerk komt) om het beste ontwerp te vinden.

In [ ]:
!pip install optuna

# GEOMETRY

## Random geometry

In [ ]:
import json
import random

# ==========================================
# 0. CONFIGURATIE VAN JOUW GEOMETRIE
# ==========================================
GRID_CELLS_X = 2          # Aantal cellen in X
GRID_CELLS_Y = 2          # Aantal cellen in Y
EDGE_LENGTH = 3.0       # Afmeting van een cel
LAYER_HEIGHT = 1.5      # Afstand tussen top en bottom layer
DIVISIONS = 8             # Aantal stappen voor de discrete verplaatsing
NUM_SAMPLES = 100       # Aantal samples
SCALE_UV = 0.25, 0.75        # Random positie in de cel

GRID = f"{GRID_CELLS_X}x{GRID_CELLS_Y}"

# ==========================================
# 1. RANDOM PARAMETERS GENEREREN (De "DNA" string)
# ==========================================
def get_random_design(json_path):
    """Leest de search space en trekt voor elke knop een willekeurige waarde."""
    with open(json_path, 'r') as f:
        search_space = json.load(f)

    random_params = {}

    for var_name, rules in search_space.items():
        if rules['type'] == 'continuous':
            # Kies een willekeurig kommagetal (bijv. voor u en v)
            random_params[var_name] = random.uniform(rules['min'], rules['max'])

        elif rules['type'] == 'discrete':
            # Kies exact één van de toegestane stapjes (bijv. voor shift_x)
            random_params[var_name] = random.choice(rules['options'])

    return random_params

# --- TEST DE FUNCTIE ---
print("Stap 1: Willekeurig DNA genereren...")
mijn_random_ontwerp = get_random_design(json_path)

print("\nSucces! Dit is het DNA van ons proef-ontwerp:")
for sleutel, waarde in mijn_random_ontwerp.items():
    print(f" - {sleutel}: \t{waarde:.3f}")

Stap 1: Willekeurig DNA genereren...

Succes! Dit is het DNA van ons proef-ontwerp:
 - v1_shift_x: 	0.375
 - v3_shift_y: 	1.125
 - v4_shift_x: 	-0.750
 - v4_shift_y: 	-0.750
 - v5_shift_y: 	0.000
 - v7_shift_x: 	-0.750
 - v9_u: 	0.263
 - v9_v: 	0.317
 - v9_shift_z: 	-1.125
 - v10_u: 	0.583
 - v10_v: 	0.319
 - v10_shift_z: 	-1.125
 - v11_u: 	0.719
 - v11_v: 	0.350
 - v11_shift_z: 	0.750
 - v12_u: 	0.384
 - v12_v: 	0.434
 - v12_shift_z: 	0.750


# Opzet

In [ ]:
import pandas as pd

# ==========================================
# 2. RECONSTRUCTIE FUNCTIES
# ==========================================
def reconstruct_vertices(params):
    """Vertaalt het DNA naar exacte 3D coördinaten (Vertices)."""
    all_vertices = []
    num_nodes_x_top = GRID_CELLS_X + 1
    num_nodes_y_top = GRID_CELLS_Y + 1

    # De 4 hoekpunten (voor de support/load classificatie)
    corners = [0, GRID_CELLS_X, GRID_CELLS_Y * (GRID_CELLS_X + 1), GRID_CELLS_Y * (GRID_CELLS_X + 1) + GRID_CELLS_X]
    top_layer_coords = {}
    vertex_idx = 0

    # --- TOP LAYER ---
    for i in range(num_nodes_y_top):
        for j in range(num_nodes_x_top):
            base_x = j * EDGE_LENGTH
            base_y = i * EDGE_LENGTH

            v_name = f"v{vertex_idx}"

            # Haal de verschuiving uit het DNA (of 0.0 als het punt niet mag schuiven)
            shift_x = params.get(f"{v_name}_shift_x", 0.0)
            shift_y = params.get(f"{v_name}_shift_y", 0.0)

            final_x = base_x + shift_x
            final_y = base_y + shift_y

            top_layer_coords[(i, j)] = {'x': final_x, 'y': final_y, 'z': 0.0}

            all_vertices.append({
                "vertex_index": v_name,
                "layer": "top",
                "x": final_x,
                "y": final_y,
                "z": 0.0
            })
            vertex_idx += 1

    # --- BOTTOM LAYER ---
    for i in range(GRID_CELLS_Y):
        for j in range(GRID_CELLS_X):
            v_name = f"v{vertex_idx}"

            # Pak de 4 hoekpunten van de cel erboven
            p_bl, p_br = top_layer_coords[(i, j)], top_layer_coords[(i, j+1)]
            p_tl, p_tr = top_layer_coords[(i+1, j)], top_layer_coords[(i+1, j+1)]

            # Haal u en v uit het DNA (of gebruik 0.5 als standaard)
            u = params.get(f"{v_name}_u", 0.5)
            v = params.get(f"{v_name}_v", 0.5)

            # Bilineaire Interpolatie wiskunde
            lx = (p_bl['x']*(1-u) + p_br['x']*u)*(1-v) + (p_tl['x']*(1-u) + p_tr['x']*u)*v
            ly = (p_bl['y']*(1-u) + p_br['y']*u)*(1-v) + (p_tl['y']*(1-u) + p_tr['y']*u)*v

            # Z positie
            final_z = -LAYER_HEIGHT + params.get(f"{v_name}_shift_z", 0.0)

            all_vertices.append({
                "vertex_index": v_name,
                "layer": "bottom",
                "x": lx,
                "y": ly,
                "z": final_z
            })
            vertex_idx += 1

    return pd.DataFrame(all_vertices)

def generate_edges(cells_x, cells_y):
    """Berekent welke punten met elkaar verbonden zijn (Edges)."""
    edges_data = []
    nodes_x_top = cells_x + 1
    nodes_y_top = cells_y + 1
    num_top = nodes_x_top * nodes_y_top
    edge_counter = 0

    def add_edge(u, v):
        nonlocal edge_counter
        edges_data.append({"edge_id": f"e{edge_counter}", "V1": u, "V2": v})
        edge_counter += 1

    # Top & Bottom connecties
    for r in range(nodes_y_top):
        for c in range(nodes_x_top):
            current = r * nodes_x_top + c
            if c < cells_x: add_edge(current, current + 1)
            if r < cells_y: add_edge(current, current + nodes_x_top)

    start_bot = num_top
    for r in range(cells_y):
        for c in range(cells_x):
            current = start_bot + r * cells_x + c
            if c < cells_x - 1: add_edge(current, current + 1)
            if r < cells_y - 1: add_edge(current, current + cells_x)

            # Diagonals
            tl, tr = r*nodes_x_top+c, r*nodes_x_top+(c+1)
            bl, br = (r+1)*nodes_x_top+c, (r+1)*nodes_x_top+(c+1)
            add_edge(current, tl); add_edge(current, tr); add_edge(current, bl); add_edge(current, br)

    return pd.DataFrame(edges_data)

# --- UITVOEREN VAN STAP 2 ---
print("Stap 2: DNA vertalen naar 3D Geometrie...")

# We gebruiken 'mijn_random_ontwerp' uit de VORIGE cel!
df_vertices = reconstruct_vertices(mijn_random_ontwerp)
df_edges = generate_edges(GRID_CELLS_X, GRID_CELLS_Y)

print(f"✅ Succes! Geometrie opgebouwd met {len(df_vertices)} knooppunten en {len(df_edges)} staven.")
print("\nVoorbeeld van de gegenereerde Vertices (eerste 5):")
print(df_vertices.head())

Stap 2: DNA vertalen naar 3D Geometrie...
✅ Succes! Geometrie opgebouwd met 13 knooppunten en 32 staven.

Voorbeeld van de gegenereerde Vertices (eerste 5):
  vertex_index layer      x      y    z
0           v0   top  0.000  0.000  0.0
1           v1   top  3.375  0.000  0.0
2           v2   top  6.000  0.000  0.0
3           v3   top  0.000  4.125  0.0
4           v4   top  2.250  2.250  0.0


## Feature extraction

In [ ]:
import math
import pandas as pd

# ==========================================
# 3. EIGENSCHAPPEN UITREKENEN (Feature Extraction)
# ==========================================
def extract_beam_properties(df_vertices, df_edges):
    """
    Berekent staaflengtes en kent constructieve profielen toe
    op basis van de positie in het ruimtevakwerk.
    """
    # Maak een snelle zoek-dictionary voor de coördinaten
    # (Dit is veel sneller dan elke keer door de hele tabel zoeken)
    v_dict = df_vertices.set_index('vertex_index').to_dict('index')
    beams = []

    for _, edge in df_edges.iterrows():
        # Let op de 'v' prefix die we in stap 2 hebben toegevoegd
        v1_id = f"v{edge['V1']}" if not str(edge['V1']).startswith('v') else edge['V1']
        v2_id = f"v{edge['V2']}" if not str(edge['V2']).startswith('v') else edge['V2']

        pt1 = v_dict[v1_id]
        pt2 = v_dict[v2_id]

        # Stelling van Pythagoras (3D Euclidische afstand in meters)
        dx = pt1['x'] - pt2['x']
        dy = pt1['y'] - pt2['y']
        dz = pt1['z'] - pt2['z']
        lengte_m = math.sqrt(dx**2 + dy**2 + dz**2)

        # Converteer naar millimeters voor de hout-database
        lengte_mm = lengte_m * 1000.0

        # Bepaal het constructieve type en de benodigde dwarsdoorsnede (in mm)
        if pt1['layer'] == 'top' and pt2['layer'] == 'top':
            b_type = 'Top Chord'
            w_req, d_req = 75.0, 150.0   # Aangepast naar beschikbare maten (75x150)
        elif pt1['layer'] == 'bottom' and pt2['layer'] == 'bottom':
            b_type = 'Bottom Chord'
            w_req, d_req = 75.0, 100.0   # Aangepast naar 75x100
        else:
            b_type = 'Diagonal Web'
            w_req, d_req = 50.0, 100.0   # Diagonalen mogen dunner zijn (50x100)

        beams.append({
            'edge_id': edge['edge_id'],
            'type': b_type,
            'Length_Req': round(lengte_mm, 2),
            'Width_Req': w_req,
            'Depth_Req': d_req
        })

    return pd.DataFrame(beams)

# --- UITVOEREN VAN STAP 3 ---
print("Stap 3: Staaflengtes en profielen berekenen...")

# We gebruiken de tabellen uit de VORIGE cel!
df_slots = extract_beam_properties(df_vertices, df_edges)

print(f"✅ Succes! We hebben de eigenschappen van {len(df_slots)} balken berekend.")
print("\nDit is de input (Slots) voor je Cost Matrix (eerste 10 balken):")
print(df_slots.head(10))

Stap 3: Staaflengtes en profielen berekenen...
✅ Succes! We hebben de eigenschappen van 32 balken berekend.

Dit is de input (Slots) voor je Cost Matrix (eerste 10 balken):
  edge_id       type  Length_Req  Width_Req  Depth_Req
0      e0  Top Chord     3375.00       75.0      150.0
1      e1  Top Chord     4125.00       75.0      150.0
2      e2  Top Chord     2625.00       75.0      150.0
3      e3  Top Chord     2515.58       75.0      150.0
4      e4  Top Chord     3000.00       75.0      150.0
5      e5  Top Chord     2928.84       75.0      150.0
6      e6  Top Chord     1875.00       75.0      150.0
7      e7  Top Chord     3824.26       75.0      150.0
8      e8  Top Chord     3750.00       75.0      150.0
9      e9  Top Chord     3000.00       75.0      150.0


## Optimalisatie

In [ ]:
# @title
# # @title
# # ==========================================
# # 3. DE OPTIMALISATIE STARTEN
# # ==========================================
# print("\n🚀 Starten met Bayesiaanse Optimalisatie...")
#
# # We vertellen Optuna dat we de score willen MINIMALISEREN (bijv. zo min mogelijk CO2/doorbuiging)
# # Wil je de 'utilization' juist maximaliseren? Gebruik dan direction='maximize'
# study = optuna.create_study(direction='minimize')
#
# # Laat de AI 100 verschillende iteraties proberen (in het echt kun je dit op 5000 zetten)
# study.optimize(objective, n_trials=100)
#
# # ==========================================
# # 4. RESULTATEN BEKIJKEN
# # ==========================================
# print("\n🏆 --- OPTIMALISATIE VOLTOOID ---")
# print(f"Beste gevonden score (bijv. kg CO2): {study.best_value:.4f}")
# print("\nBeste Geometrie (Parameter Vector) om in te laden in Grasshopper:")
#
# # Print de ideale parameters die Optuna heeft gevonden
# for var_name, best_val in study.best_params.items():
#     print(f"  - {var_name}: \t {best_val:.3f}")

# COST MATRIX

Omdat de pseudo-LCA berekening (uit de vorige stap) de impact van de gehele balk al berekent, heb je nu een methodologische keuze:

Optie A (De dubbele boete): Je telt de LCA-score én de penalty's bij elkaar op. Hiermee straf je afval extra zwaar af ten opzichte van het puur inbrengen van materiaal. Dit dwingt het algoritme agressief naar optimaal materiaalgebruik.

Optie B (De uitsplitsing): Je berekent de LCA-score alleen over het nuttige deel, en gebruikt je nieuwe formules om het verlies in kaart te brengen.

In [ ]:
# ==========================================
# CEL 1: INPUTS EN ALGEMENE PARAMETERS
# ==========================================
import pandas as pd
import numpy as np

# GWP Waarden (kg CO2 eq / kg hout)
GWP_VIRGIN = 0.50
GWP_RECLAIMED = 0.08

# LCA Parameters voor E_cost
PROCESSING_PENALTY_CO2 = 5.0  # kg CO2 boete voor bewerkingen (bijv. ontspijkeren)

print("✅ Parameters succesvol geladen.")

✅ Parameters succesvol geladen.


In [ ]:
# ==========================================
# CEL 2: DE REKENFUNCTIES (MODULES)
# ==========================================

def calculate_pseudo_lca_stock(df_stock):
    """
    Berekent de pseudo-LCA score (E_cost) voor een reeds ingeladen DataFrame
    met de timber stock.
    """
    # Maak een kopie om waarschuwingen (SettingWithCopyWarning) te voorkomen
    # en de originele input dataset zuiver te houden.
    df_lca = df_stock.copy()

    print("Start in-memory pseudo-LCA berekeningen...")

    # Stap A: Bereken Volume in m3 (dimensies zijn in mm, dus delen door 1000)
    df_lca['Volume_m3'] = (df_lca['Length_Actual'] / 1000) * (df_lca['Width'] / 1000) * (df_lca['Depth'] / 1000)

    # Stap B: Material Impact (A1-A3)
    df_lca['Impact_Material_kgCO2'] = df_lca['Volume_m3'] * df_lca['Embodied Carbon Coëfficiënt']

    # Stap C: Transport Impact (A4)
    # Dichtheid (Density) delen door 1000 om van kg/m3 naar ton/m3 te gaan
    df_lca['Impact_Transport_kgCO2'] = df_lca['Volume_m3'] * (df_lca['Density'] / 1000) * df_lca['Transport_Dist'] * df_lca['Emmisiefactor']

    # Stap D: Processing Impact (C3 / A3 Re-processing)
    # Binaire bewerkingsfactor (0 of 1) maal de vaste penalty
    df_lca['Impact_Processing_kgCO2'] = df_lca['Bewerkingsfactor'] * PROCESSING_PENALTY_CO2

    # Stap E: Totale E_cost Berekenen
    df_lca['E_cost_Total_kgCO2'] = (
        df_lca['Impact_Material_kgCO2'] +
        df_lca['Impact_Transport_kgCO2'] +
        df_lca['Impact_Processing_kgCO2']
    )

    return df_lca


def calculate_geometric_penalties(slot, stock_item):
    """
    Module 2: Berekent de CO2-penalty's voor zaagverlies en overdimensionering
    voor één specifieke match tussen een Slot (ontwerp) en Stock (voorraad).
    Geeft een tuple terug: (c_waste, c_overdim).
    """
    # 1. Haal afmetingen op en converteer naar meters
    l_slot = slot['Length_Req'] / 1000.0
    w_req = slot['Width_Req'] / 1000.0
    d_req = slot['Depth_Req'] / 1000.0
    a_req = w_req * d_req

    l_stock = stock_item['Length_Actual'] / 1000.0
    w_stock = stock_item['Width'] / 1000.0
    d_stock = stock_item['Depth'] / 1000.0
    a_stock = w_stock * d_stock

    rho = stock_item['Density'] # kg/m3

    # Status: 0 = Virgin, 1 = Reclaimed (afhankelijk van hoe je dataset in elkaar zit, pas dit evt. aan)
    gwp_unit = GWP_RECLAIMED if stock_item['State'] == 1 else GWP_VIRGIN

    # Zaagverlies en Overdimensionering in kg CO2 eq
    c_waste = (l_stock - l_slot) * a_stock * rho * gwp_unit
    c_overdim = (a_stock - a_req) * l_slot * rho * gwp_unit

    return max(0, c_waste), max(0, c_overdim)

print("✅ Rekenmodules succesvol gedefinieerd.")

✅ Rekenmodules succesvol gedefinieerd.


## Controle

In [ ]:
# @title
# 1. Voer de functie uit op je reeds ingeladen DataFrame
df_stock_with_lca = calculate_pseudo_lca_stock(df_input_stock)

# 2. Bekijk de resultaten van de berekening
print("\nPreview van de berekende E_cost per element:")
display(df_stock_with_lca.sample(10))

Start in-memory pseudo-LCA berekeningen...

Preview van de berekende E_cost per element:


,Member_ID,State,Length_Actual,Depth,Width,E_modulus_eff,f_mk,Density,Embodied Carbon Coëfficiënt,Transport_Dist,Emmisiefactor,Bewerkingsfactor,Volume_m3,Impact_Material_kgCO2,Impact_Transport_kgCO2,Impact_Processing_kgCO2,E_cost_Total_kgCO2
100,NS_00100,0,3300.0,225.0,38.0,11000.0,24,420,150.0,500,0.1045,0,0.028215,4.23225,0.619178,0.0,4.851428
0,NS_00000,0,2400.0,100.0,38.0,11000.0,24,420,150.0,500,0.1473,0,0.009120,1.36800,0.282109,0.0,1.650109
19,NS_00019,0,2400.0,225.0,100.0,11000.0,24,420,150.0,500,0.1426,0,0.054000,8.10000,1.617084,0.0,9.717084
109,NS_00109,0,3300.0,300.0,50.0,11000.0,24,420,150.0,500,0.0980,0,0.049500,7.42500,1.018710,0.0,8.443710
123,NS_00123,0,3600.0,175.0,100.0,11000.0,24,420,150.0,500,0.1012,0,0.063000,9.45000,1.338876,0.0,10.788876
162,NS_00162,0,4000.0,250.0,75.0,11000.0,24,420,150.0,500,0.1417,0,0.075000,11.25000,2.231775,0.0,13.481775
137,NS_00137,0,3600.0,300.0,50.0,11000.0,24,420,150.0,500,0.0999,0,0.054000,8.10000,1.132866,0.0,9.232866
111,NS_00111,0,3300.0,300.0,100.0,11000.0,24,420,150.0,500,0.0842,0,0.099000,14.85000,1.750518,0.0,16.600518
81,NS_00081,0,3000.0,300.0,50.0,11000.0,24,420,150.0,500,0.1425,0,0.045000,6.75000,1.346625,0.0,8.096625
75,NS_00075,0,3000.0,225.0,100.0,11000.0,24,420,150.0,500,0.0813,0,0.067500,10.12500,1.152428,0.0,11.277428


## Building of cost matrix

In [ ]:
# Print even ter controle hoeveel balken erin zitten
print(f"🪵 Hout-inventaris succesvol ingeladen: {len(df_input_stock)} balken beschikbaar.")

🪵 Hout-inventaris succesvol ingeladen: 184 balken beschikbaar.


In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# CEL 3: HOOFDSCRIPT - BOUW DE COST MATRIX
# ==========================================

def build_optimization_matrix(df_design, df_stock_raw):
    print("Start generatie van de integrale CO2 Cost Matrix...")

    # 1. Bereken de basis LCA score (Roep de module uit Cel 2 aan)
    df_stock = calculate_pseudo_lca_stock(df_stock_raw)

    n_slots = len(df_design)
    n_stock = len(df_stock)

    # Gebruik np.inf (oneindig) voor plekken waar het niet past.
    # Dit is perfect voor het latere toewijzingsalgoritme.
    cost_matrix = np.full((n_slots, n_stock), np.inf)

    # Bijhouden hoeveel matches er daadwerkelijk lukken
    succesvolle_matches = 0

    for i in range(n_slots):
        slot = df_design.iloc[i]

        # Bepaal de grootste en kleinste doorsnedemaat van het SLOT (voor rotatie)
        slot_max_dim = max(slot['Width_Req'], slot['Depth_Req'])
        slot_min_dim = min(slot['Width_Req'], slot['Depth_Req'])

        for j in range(n_stock):
            stock_item = df_stock.iloc[j]

            # Bepaal de grootste en kleinste doorsnedemaat van de STOCK
            stock_max_dim = max(stock_item['Width'], stock_item['Depth'])
            stock_min_dim = min(stock_item['Width'], stock_item['Depth'])

            # --- HARD CONSTRAINTS (INCLUSIEF ROTATIE) ---
            if (stock_item['Length_Actual'] >= slot['Length_Req'] and
                stock_max_dim >= slot_max_dim and
                stock_min_dim >= slot_min_dim):

                # Het past! Haal de basis E_cost op uit de LCA module
                # (Aangepast naar 'E_cost_Total_kgCO2' op basis van je Cel 2 code)
                e_cost_base = stock_item['E_cost_Total_kgCO2']

                # Roep Module 2 aan voor penalty's
                c_waste, c_overdim = calculate_geometric_penalties(slot, stock_item)

                # Totale score
                total_match_score = e_cost_base + c_waste + c_overdim
                cost_matrix[i, j] = total_match_score
                succesvolle_matches += 1

    print(f"✅ Matrix gegenereerd! Dimensies: {n_slots} benodigde staven x {n_stock} inventaris-balken.")
    print(f"📊 Aantal fysiek geldige combinaties gevonden: {succesvolle_matches}")

    if succesvolle_matches == 0:
        print("\n⚠️ WAARSCHUWING: 0 matches gevonden. Je voorraad bevat geen elementen die groot genoeg zijn voor je ontwerp.")
        print(f"Grootste element in voorraad: Lengte={df_stock['Length_Actual'].max()}mm, Breedte={df_stock['Width'].max()}mm, Diepte={df_stock['Depth'].max()}mm")

    return cost_matrix, df_stock

# --- UITVOEREN VAN DE WORKFLOW MET ECHTE DATA ---
# 2. Bouw de matrix!
# We gebruiken hier 'df_slots' uit jouw vorige script (de gegenereerde geometrie).
cost_matrix, verrijkte_stock = build_optimization_matrix(df_slots, df_input_stock)

# 3. Presentatie van het resultaat
# Maak er een DataFrame van, puur om het netjes te kunnen printen.
df_cost_matrix_display = pd.DataFrame(
    cost_matrix,
    index=[f"{row['edge_id']} ({row['Length_Req']:.0f}mm)" for _, row in df_slots.iterrows()],
    columns=verrijkte_stock['Member_ID'].tolist()
)
# 4. Opslaan van resultaat
df_cost_matrix_display.to_csv('final_cost_matrix.csv', index=True)

print("\nPreview Cost Matrix (eerste 8 staven, eerste 5 inventaris-balken):")
print("(Let op: 'inf' betekent dat de inventaris-balk te klein is en wordt uitgesloten)")
print("-" * 60)
print(df_cost_matrix_display.iloc[:8, :5].round(2))

Start generatie van de integrale CO2 Cost Matrix...
Start in-memory pseudo-LCA berekeningen...
✅ Matrix gegenereerd! Dimensies: 32 benodigde staven x 184 inventaris-balken.
📊 Aantal fysiek geldige combinaties gevonden: 2122

Preview Cost Matrix (eerste 8 staven, eerste 5 inventaris-balken):
(Let op: 'inf' betekent dat de inventaris-balk te klein is en wordt uitgesloten)
------------------------------------------------------------
             NS_00000  NS_00001  NS_00002  NS_00003  NS_00004
e0 (3375mm)       inf       inf       inf       inf       inf
e1 (4125mm)       inf       inf       inf       inf       inf
e2 (2625mm)       inf       inf       inf       inf       inf
e3 (2516mm)       inf       inf       inf       inf       inf
e4 (3000mm)       inf       inf       inf       inf       inf
e5 (2929mm)       inf       inf       inf       inf       inf
e6 (1875mm)       inf       inf       inf       inf       inf
e7 (3824mm)       inf       inf       inf       inf       inf


# MATCHING ALGORITHM / ILP

## Script

In [ ]:
import pulp
import numpy as np
import pandas as pd

# ==========================================
# CEL 4: TOEPASSING VAN HET OPTIMALISATIE ALGORITME (MILP)
# ==========================================
print("Start MILP Optimizer voor definitieve toewijzing...")

# 1. DATA KOPPELEN VANUIT VORIGE SCRIPTS
# Haal de namen op uit de DataFrames van de vorige cellen
stock_items = verrijkte_stock['Member_ID'].tolist()
construction_slots = df_slots['edge_id'].tolist()

# Dynamische categorisatie (kijkt naar 'NS' en 'RS' in de Member_ID)
new_items = [item for item in stock_items if 'NS' in item]
reclaimed_items = [item for item in stock_items if 'RS' in item]

print(f"📊 Inventaris: {len(reclaimed_items)} Reclaimed, {len(new_items)} New elementen.")
print(f"📐 Constructie vereist: {len(construction_slots)} staven.")

# 2. FILTEREN VAN DE COST MATRIX (Sparse Matrix Maken)
# We maken alleen combinaties aan die fysiek passen (waar de cost NIET oneindig is)
valid_matches = []
costs = {}

for i, slot_id in enumerate(construction_slots):
    for j, stock_id in enumerate(stock_items):
        cost = cost_matrix[i, j]
        # Als de cost niet 'inf' is, is het een geldige match!
        if cost != np.inf:
            valid_matches.append((stock_id, slot_id))
            costs[(stock_id, slot_id)] = cost

print(f"⚡ Aantal geldige combinaties voor de solver gereduceerd tot: {len(valid_matches)}")

# 3. HET MODEL OPZETTEN
prob = pulp.LpProblem("Reclaimed_Timber_Matching", pulp.LpMinimize)

# 4. DECISION VARIABLES
# We maken uitsluitend 'knoppen' aan voor de combinaties die kunnen passen
x = pulp.LpVariable.dicts("Match", valid_matches, 0, 1, pulp.LpBinary)

# 5. OBJECTIVE FUNCTION (Doel: Minimaliseer de totale CO2 penalty)
prob += pulp.lpSum([x[match] * costs[match] for match in valid_matches])

# 6. CONSTRAINTS (De Regels)

# Regel A: Elke staaf in de constructie MOET precies 1 balk toegewezen krijgen
for slot_id in construction_slots:
    # Zoek alle balken die überhaupt passen op dit slot
    valid_stock_for_slot = [stock_id for (stock_id, s_id) in valid_matches if s_id == slot_id]

    if not valid_stock_for_slot:
        print(f"⚠️ FATALE FOUT: Slot {slot_id} heeft GEEN ENKELE fysiek passende balk in de hele voorraad!")
    else:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for stock_id in valid_stock_for_slot]) == 1

# Regel B: Reclaimed hout is uniek (Max 1x gebruiken)
for stock_id in reclaimed_items:
    valid_slots_for_stock = [s_id for (s_id_tuple, s_id) in valid_matches if s_id_tuple == stock_id]
    if valid_slots_for_stock:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for slot_id in valid_slots_for_stock]) <= 1

# Regel C: Nieuw hout mag oneindig gebruikt worden (Limiet = aantal benodigde staven)
for stock_id in new_items:
    valid_slots_for_stock = [s_id for (s_id_tuple, s_id) in valid_matches if s_id_tuple == stock_id]
    if valid_slots_for_stock:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for slot_id in valid_slots_for_stock]) <= len(construction_slots)

# ==========================================
# 7. OPLOSSEN EN RESULTATEN
# ==========================================
prob.solve()

print("\n" + "="*50)
print(f"STATUS OPLOSSING: {pulp.LpStatus[prob.status]}")
print("="*50)

if pulp.LpStatus[prob.status] == 'Optimal':
    total_cost = pulp.value(prob.objective)

    # Sla de winnende combinaties op in een overzichtelijke lijst
    results = []
    for j in construction_slots:
        for i in stock_items:
            match = (i, j)
            if match in x and x[match].varValue == 1:
                results.append({'Slot_ID': j, 'Assigned_Stock': i, 'CO2_Penalty': round(costs[match], 2)})

    df_results = pd.DataFrame(results)

    print(f"\n✅ Het optimale ontwerp is gevonden met een totale CO2 penalty van {total_cost:.2f} kg!")
    print("\nDefinitieve Hout-Toewijzing:")
    print("-" * 50)
    print(df_results.to_string(index=False))

    # Optioneel: Exporteer dit naar CSV om in Grasshopper te kleuren
    # df_results.to_csv('definitieve_toewijzing.csv', index=False)
else:
    print("❌ Het algoritme kon geen oplossing vinden. Waarschijnlijk is je voorraad niet toereikend voor de opgevraagde constructie.")

Start MILP Optimizer voor definitieve toewijzing...
📊 Inventaris: 16 Reclaimed, 168 New elementen.
📐 Constructie vereist: 32 staven.
⚡ Aantal geldige combinaties voor de solver gereduceerd tot: 2122

STATUS OPLOSSING: Optimal

✅ Het optimale ontwerp is gevonden met een totale CO2 penalty van 198.92 kg!

Definitieve Hout-Toewijzing:
--------------------------------------------------
Slot_ID Assigned_Stock  CO2_Penalty
     e0       NS_00118         7.74
     e1       RS_00002        55.98
     e2       NS_00034         5.48
     e3       NS_00034         5.74
     e4       NS_00062         5.83
     e5       NS_00062         6.00
     e6       NS_00006         6.01
     e7       NS_00146         8.22
     e8       NS_00146         8.39
     e9       NS_00062         5.83
    e10       NS_00006         5.12
    e11       NS_00146         8.39
    e12       NS_00142         5.05
    e13       NS_00114         4.82
    e14       NS_00057         2.60
    e15       NS_00141         3.62
   

# EXPORT

This is where the best parameters of the sctructure are exported, this is done in a format that can be used by grasshopper script to translate it into geometry

In [ ]:
# @title
def generate_edges(num_samples, cells_x, cells_y):
    edges_data = []

    # Bereken hulpparameters
    nodes_x_top = cells_x + 1
    nodes_y_top = cells_y + 1
    num_top_vertices = nodes_x_top * nodes_y_top

    # We itereren door elke sample om de edges per sample vast te leggen
    for sample_id in range(num_samples):

        edge_counter = 0  # Reset edge counter per sample (of wil je unieke ID's over de hele file? Meestal per sample resetten: e0..e127)

        # Hulpfunctie om edge toe te voegen
        def add_edge(u, v):
            nonlocal edge_counter
            edges_data.append({
                "sample_id": sample_id,
                "edge_id": f"e{edge_counter}",
                "V1": u,
                "V2": v,
            })
            edge_counter += 1

        # --- 1. TOP LAYER GRID ---
        # Vertices 0 tot num_top_vertices-1
        for r in range(nodes_y_top):      # loop rijen
            for c in range(nodes_x_top):  # loop kolommen
                current = r * nodes_x_top + c

                # Horizontaal (naar rechts)
                if c < cells_x: # zolang niet de laatste kolom
                    add_edge(current, current + 1)

                # Verticaal (naar beneden, of 'boven' in matrix index)
                if r < cells_y: # zolang niet de laatste rij
                    add_edge(current, current + nodes_x_top)

        # --- 2. BOTTOM LAYER GRID ---
        # Start index is na de laatste top vertex
        start_idx_bottom = num_top_vertices

        # Bottom grid heeft evenveel punten als er cellen zijn (cells_x * cells_y)
        # Maar de grid verbindingen zijn er eentje minder dan het aantal punten
        # Bottom punten zijn een grid van (cells_x) breed bij (cells_y) hoog.

        for r in range(cells_y):
            for c in range(cells_x):
                current = start_idx_bottom + r * cells_x + c

                # Horizontaal (naar rechts)
                if c < cells_x - 1:
                    add_edge(current, current + 1)

                # Verticaal (naar beneden)
                if r < cells_y - 1:
                    add_edge(current, current + cells_x)

        # --- 3. DIAGONALS (Pyramid connections) ---
        # Verbind elke Bottom vertex met de 4 Top vertices erboven
        for r in range(cells_y):
            for c in range(cells_x):
                bottom_node = start_idx_bottom + r * cells_x + c

                # De 4 corresponderende punten in de Top layer
                # Top grid is (cells_x + 1) breed
                top_tl = r * nodes_x_top + c               # Top-Left (of row i)
                top_tr = r * nodes_x_top + (c + 1)         # Top-Right
                top_bl = (r + 1) * nodes_x_top + c         # Bottom-Left (row i+1)
                top_br = (r + 1) * nodes_x_top + (c + 1)   # Bottom-Right

                add_edge(bottom_node, top_tl)
                add_edge(bottom_node, top_tr)
                add_edge(bottom_node, top_bl)
                add_edge(bottom_node, top_br)

    return pd.DataFrame(edges_data)

In [ ]:
# @title
import pandas as pd

# ==========================================
# 5. GEOMETRISCHE RECONSTRUCTIE VAN HET OPTIMUM
# ==========================================
print("\n📦 Exporteren van het volledige winnende ontwerp naar CSV's...")

# Haal de winnende getallen op (de 'DNA' vector)
best_parameters = study.best_params
OPTIMUM_ID = 9999

def reconstruct_optimum_vertices(params, sample_id):
    """Herbouwt de exacte coördinaten op basis van de Optuna parameters."""
    all_vertices = []
    num_nodes_x_top = GRID_CELLS_X + 1
    num_nodes_y_top = GRID_CELLS_Y + 1

    # Bepaal hoeken voor de support/load logica
    def get_corner_indices(cells_x, cells_y):
        return {
            "bl": 0,
            "br": cells_x,
            "tl": cells_y * (cells_x + 1),
            "tr": cells_y * (cells_x + 1) + cells_x
        }
    corners = get_corner_indices(GRID_CELLS_X, GRID_CELLS_Y).values()

    top_layer_coords = {}
    vertex_idx = 0

    # --- STAP 1: TOP LAYER ---
    for i in range(num_nodes_y_top):
        for j in range(num_nodes_x_top):
            base_x = j * EDGE_LENGTH
            base_y = i * EDGE_LENGTH
            base_z = 0.0

            attribute = "support" if vertex_idx in corners else "load"
            v_name = f"v{vertex_idx}"

            # Haal de specifieke shift op uit de best_parameters.
            # Als hij niet in de dictionary staat (omdat Optuna hem niet als variabele had), is de shift 0.0
            shift_x = params.get(f"{v_name}_shift_x", 0.0)
            shift_y = params.get(f"{v_name}_shift_y", 0.0)

            final_x = base_x + shift_x
            final_y = base_y + shift_y
            final_z = base_z

            top_layer_coords[(i, j)] = {'x': final_x, 'y': final_y, 'z': final_z}

            all_vertices.append({
                "sample_id": sample_id,
                "vertex_index": v_name,
                "layer": "top",
                "attribute": attribute,
                "x": round(final_x, 3),
                "y": round(final_y, 3),
                "z": round(final_z, 3)
            })
            vertex_idx += 1

    # --- STAP 2: BOTTOM LAYER ---
    # def bilinear_interpolate(...) # Plak hierboven eventueel je interpolatie functie als hij niet globaal staat
    for i in range(GRID_CELLS_Y):
        for j in range(GRID_CELLS_X):
            v_name = f"v{vertex_idx}"

            p_bl = top_layer_coords[(i, j)]
            p_br = top_layer_coords[(i, j+1)]
            p_tl = top_layer_coords[(i+1, j)]
            p_tr = top_layer_coords[(i+1, j+1)]

            # Haal de u en v op uit de Optuna dictionary
            u = params.get(f"{v_name}_u", 0.5)
            v = params.get(f"{v_name}_v", 0.5)

            # Bilineaire Interpolatie wiskunde
            x_bot = p_bl['x'] * (1 - u) + p_br['x'] * u
            x_top = p_tl['x'] * (1 - u) + p_tr['x'] * u
            lx = x_bot * (1 - v) + x_top * v

            y_bot = p_bl['y'] * (1 - u) + p_br['y'] * u
            y_top = p_tl['y'] * (1 - u) + p_tr['y'] * u
            ly = y_bot * (1 - v) + y_top * v

            # Z positie
            z_shift = params.get(f"{v_name}_shift_z", 0.0)
            final_z = -LAYER_HEIGHT + z_shift

            all_vertices.append({
                "sample_id": sample_id,
                "vertex_index": v_name,
                "layer": "bottom",
                "attribute": "hinges",
                "x": round(lx, 3),
                "y": round(ly, 3),
                "z": round(final_z, 3)
            })
            vertex_idx += 1

    return pd.DataFrame(all_vertices)

# --- 2. Genereer de Vertices ---
df_opt_vertices = reconstruct_optimum_vertices(best_parameters, OPTIMUM_ID)

# --- 3. Genereer de Edges ---
# Je bestaande generate_edges() functie is PERFECT, want die heeft de parameters niet eens nodig!
# De topologie (welk punt zit aan welk punt) is namelijk voor elk ontwerp hetzelfde.
# We roepen hem simpelweg 1x aan voor dit sample.
df_opt_edges = generate_edges(num_samples=1, cells_x=GRID_CELLS_X, cells_y=GRID_CELLS_Y)

# BELANGRIJK: Omdat generate_edges() altijd bij sample_id=0 begint,
# forceren we de ID naar ons OPTIMUM_ID, zodat hij synchroon loopt met de vertices.
df_opt_edges['sample_id'] = OPTIMUM_ID

# --- 4. Exporteren ---
pad_vertices = 'optimum_vertices.csv'
pad_edges = 'optimum_edges.csv'

df_opt_vertices.to_csv(pad_vertices, index=False)
df_opt_edges.to_csv(pad_edges, index=False)

print(f"✅ Succes! Volledig winnende geometrie ({len(df_opt_vertices)} punten, {len(df_opt_edges)} lijnen) opgeslagen.")
print(f"   -> {pad_vertices}")
print(f"   -> {pad_edges}")